In [ ]:
import os
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# =========================
# CPU OPTIMIZATION
# =========================
# Must be set before heavy PyTorch work starts
NUM_CORES = os.cpu_count() or 4
torch.set_num_threads(NUM_CORES)
torch.set_num_interop_threads(max(1, min(4, NUM_CORES)))  # keep interop modest on CPU

# Optional env vars for BLAS/OpenMP-backed installs
os.environ["OMP_NUM_THREADS"] = str(NUM_CORES)
os.environ["MKL_NUM_THREADS"] = str(NUM_CORES)

# =========================
# CONFIG
# =========================
MODEL_NAME = "facebook/nllb-200-distilled-600M"

SRC_LANG = "npi_Deva"   # Nepali
TGT_LANG = "eng_Latn"   # English

TEXT_COL = "text"

# Input files
TRAIN_PATH = "twitter_train.csv"
VAL_PATH   = "twitter_val.csv"
TEST_PATH  = "twitter_test.csv"

# Output files
OUT_TRAIN_PATH = "twitter_train_en.csv"
OUT_VAL_PATH   = "twitter_val_en.csv"
OUT_TEST_PATH  = "twitter_test_en.csv"

# Speed settings for tweets
BATCH_SIZE = 32      # try 64 if RAM allows
MAX_INPUT_LENGTH = 96
MAX_NEW_TOKENS = 96

# =========================
# LOAD MODEL
# =========================
device = "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang=SRC_LANG)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

print(f"Using device: {device}")
print(f"CPU cores detected: {NUM_CORES}")
print(f"PyTorch intra-op threads: {torch.get_num_threads()}")
print(f"PyTorch inter-op threads: {torch.get_num_interop_threads()}")

# NLLB target language token id
forced_bos_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)

def clean_batch_text(batch):
    cleaned = []
    for x in batch:
        if pd.isna(x):
            cleaned.append("")
        else:
            text = str(x).strip()
            cleaned.append(text)
    return cleaned

def translate_text(texts, batch_size=BATCH_SIZE):
    translated_texts = []

    # Make sure tokenizer knows source language
    tokenizer.src_lang = SRC_LANG

    for i in tqdm(range(0, len(texts), batch_size), desc="Translating"):
        batch = texts[i:i + batch_size]
        batch = clean_batch_text(batch)

        # Preserve empties without sending them through generation
        nonempty_indices = [idx for idx, txt in enumerate(batch) if txt]
        batch_outputs = [""] * len(batch)

        if nonempty_indices:
            nonempty_texts = [batch[idx] for idx in nonempty_indices]

            inputs = tokenizer(
                nonempty_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_INPUT_LENGTH
            ).to(device)

            with torch.no_grad():
                generated_tokens = model.generate(
                    **inputs,
                    forced_bos_token_id=forced_bos_token_id,
                    num_beams=1,
                    do_sample=False,
                    max_new_tokens=MAX_NEW_TOKENS,
                    early_stopping=True
                )

            decoded = tokenizer.batch_decode(
                generated_tokens,
                skip_special_tokens=True
            )

            for idx, out in zip(nonempty_indices, decoded):
                batch_outputs[idx] = out

        translated_texts.extend(batch_outputs)

    return translated_texts

def translate_file(input_path, output_path, text_col=TEXT_COL):
    print(f"\nReading: {input_path}")
    df = pd.read_csv(input_path)

    if text_col not in df.columns:
        raise ValueError(
            f"Column '{text_col}' not found in {input_path}. "
            f"Available columns: {list(df.columns)}"
        )

    texts = df[text_col].tolist()
    translated = translate_text(texts)

    df[f"{text_col}_en"] = translated
    df.to_csv(output_path, index=False, encoding="utf-8")

    print(f"Saved: {output_path}")
    print(df[[text_col, f"{text_col}_en"]].head())

# =========================
# RUN ALL SPLITS
# =========================
translate_file(TRAIN_PATH, OUT_TRAIN_PATH)
translate_file(VAL_PATH, OUT_VAL_PATH)
translate_file(TEST_PATH, OUT_TEST_PATH)

print("\nDone. All three splits translated.")

/opt/anaconda3/envs/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
CPU cores detected: 10
PyTorch intra-op threads: 10
PyTorch inter-op threads: 4

Reading: twitter_train.csv


Translating:  14%|█▎        | 100/732 [21:56<2:33:31, 14.57s/it]